In [2]:
# LESHI* - Line Emission Source Hunting Integrator
# written by Michalina Maksymowicz-Maciata

# *Leshi (also known as Leshy or Leshen) is a tutelary deity of the forest and hunting in Slavic mythology.

##########CONTROL PANEL#######################################################
file =  "MIGHTEE-HI_DR1_COSMOS_L2_r0p0_clean_conv_3001-4055.fits"
path_to_data = "../data/"
path_to_results = "../results_inverted/"

int_image_length = 10 # length in channels of the cube slab to create moment 0 map from (depends on spectral resolution, should be approximately the length of expected signals in channels)
dithering = True
background_noise_unifrom = False

SNR_integrated_image_threshold = 3
SNR_channel_frame_threshold = 2.5
SNR_spectrum_threshold = 2.5
rsqr_threshold = 0.1

max_signal_jump = 6 # depends on data spatial and spectral resolution
signal_persistence_threshold = 2 # depends on data spectral resolution

min_dist = 2 # smallest expected distance in beamsize radiuses between two galaxies at close frequency

cube_start = 0 # range of the cube to search through
cube_end = 100 #put None if cube is to contain the last element of the datacube

##############################################################################

In [4]:
# imports
import numpy as np
import pandas as pd
import emcee
from scipy.optimize import curve_fit
import os
import copy
import multiprocessing

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib


from astropy.io import fits  # We use fits to open the actual data file
from astropy.stats import sigma_clipped_stats, sigma_clip
from astropy.wcs import WCS
from astropy.table import Table
from astropy.coordinates import SkyCoord

from photutils.detection import find_peaks
from photutils.aperture import CircularAperture, RectangularAperture, CircularAnnulus
from photutils.aperture import aperture_photometry

import time
start = time.time()
import warnings
warnings.filterwarnings('ignore')

In [5]:
# functions for fitting gaussian function to the spectrum
def log_likelihood(theta, x, y, yerr):
    H, A, x0, sigma = theta
    model = H + A * np.exp(-(x - x0) ** 2 / (2 * sigma ** 2))

    sigma2 = yerr**2
    return -0.5 * np.sum((y - model) ** 2 / sigma2 + np.log(sigma2))

def log_prior(theta):
    H, A, x0, sigma = theta
    if 0.0 < A and 0 < x0 < cube_wavelength_range and 0.00001 < sigma < 1000:
        return 0.0
    return -np.inf

def log_probability(theta, x, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, x, y, yerr)

def gauss(x, H, A, x0, sigma): 
            return H + A * np.exp(-(x - x0) ** 2 / (2 * sigma ** 2))
    
def fit_gauss(x,y):

    #try:
    pos = [0,np.max(y),x[np.argmax(y)],5] + 1e-5 * np.random.randn(32, 4)
    
    nwalkers, ndim = pos.shape
    yerr=0.0001
    
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=(x, y, yerr))
    sampler.run_mcmc(pos, 3000, progress=False);
    flat_samples = sampler.get_chain(discard=100, thin=15, flat=True)

    fit_H = np.percentile(flat_samples[:, 0], [50])
    fit_A = np.percentile(flat_samples[:, 1], [50])
    fit_x0 = np.percentile(flat_samples[:, 2], [50])
    fit_sigma = np.percentile(flat_samples[:, 3], [50])
    fit_theta = [fit_H,fit_A,fit_x0,fit_sigma]
    
    fit_y = gauss(x, fit_H, fit_A, fit_x0, fit_sigma) 

    rsqr = 1- (np.sum((y-fit_y)**2))/(np.sum((y-np.sum(y)/len(y))**2))
    # print('Rsqr: ',rsqr)
    return rsqr, fit_x0[0], fit_sigma[0], fit_A[0], fit_H[0]


# get sky coordinates in degrees from pixel coordinates
def sky_coord_in_deg_from_pix(x_pix_coord,y_pix_coord,wcs):
    skycoord = SkyCoord.from_pixel(xp=x_pix_coord,yp=y_pix_coord, wcs=wcs)
    skycoord_ra = str(skycoord.ra)
    a = np.array([skycoord_ra.split('d')[0], ((skycoord_ra.split('d')[1]).split('m'))[0], (((skycoord_ra.split('d')[1]).split('m'))[1]).split('s')[0]])
    a = a.astype(float)
    RA = a[0]+a[1]/60+a[2]/3600
    
    skycoord_dec = str(skycoord.dec)
    a = np.array([skycoord_dec.split('d')[0], ((skycoord_dec.split('d')[1]).split('m'))[0], (((skycoord_dec.split('d')[1]).split('m'))[1]).split('s')[0]])
    a = a.astype(float)
    DEC = a[0]+a[1]/60+a[2]/3600

    return RA, DEC

# get frequency form channel
def channel_to_frequency(channel,wave0,wavedelta,channel0):
    frequency = wave0+wavedelta*(channel-channel0)
    return frequency


# functions for calculating stadard deviation of the background for different radiuses from the center of the image

def threshold_for_circle(image_slice,cent_coord,rout,threshold_std):
    aperture = CircularAperture(cent_coord, r=rout)
    mask= aperture.to_mask(method='center')
    image_slice = mask.multiply(image_slice,fill_value=np.nan)   
    mean, median, std = sigma_clipped_stats(image_slice, sigma=3.0)
    threshold = median + (threshold_std * std)
    return threshold, median, std

def exp_function(x,a,b,c,d):
    y = a*np.exp(b*x+d)+c
    return y

def log_likelihood_exp(theta, x, y, yerr):
    model = exp_function(x,*theta)
    sigma2 = yerr**2
    return -0.5 * np.sum((y - model) ** 2 / sigma2 + np.log(sigma2))

def log_prior_exp(theta):
    a,b,c,d = theta
    if b>0 and a>0 and 0<d<1:
        return 0.0
    return -np.inf

def log_probability_exp(theta, x, y, yerr):
    lp = log_prior_exp(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_exp(theta, x, y, yerr)
  
def fit_exp(x,y):

    #try:
    pos = [y[0],5,y[0],0] + 1e-5 * np.random.randn(32, 4)
    
    nwalkers, ndim = pos.shape
    yerr=0.000001
    
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability_exp, args=(x, y, yerr))
    sampler.run_mcmc(pos, 3000, progress=False);
    flat_samples = sampler.get_chain(discard=100, thin=15, flat=True)

    fit_a = np.percentile(flat_samples[:, 0], [50])
    fit_b = np.percentile(flat_samples[:, 1], [50])
    fit_c = np.percentile(flat_samples[:, 2], [50])
    fit_d = np.percentile(flat_samples[:, 3], [50])
    fit_theta = (fit_a,fit_b,fit_c,fit_d)

    return fit_theta

def std_and_median_from_radius_function(image,cent_coord,image_radius):
    rin_array = np.array([0.2,0.4,0.6,0.8])
    rout_array = np.array([0.21,0.41,0.61,0.81])

    std_array=np.array([])
    median_array=np.array([])
    for r in range(4):
        rin = rin_array[r]*image_radius
        rout = rout_array[r]*image_radius
        
        aperture_annulus = CircularAnnulus(cent_coord, rin,rout)
        mask= aperture_annulus.to_mask(method='center')
        image_slice = mask.multiply(image,fill_value=np.nan)
        
        mean, median, std = sigma_clipped_stats(image_slice, sigma=3.0)
        std_array = np.append(std_array,std)
        median_array = np.append(median_array,median)
    popt_std = fit_exp(rout_array, std_array)
    median = np.mean(median_array)
    return popt_std, median

def create_circular_mask(h, w, center=None, radius=None):

    if center is None: # use the middle of the image
        center = (int(w/2), int(h/2))
    if radius is None: # use the smallest distance between the center and image walls
        radius = min(center[0], center[1], w-center[0], h-center[1])

    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - center[0])**2 + (Y-center[1])**2)

    mask = dist_from_center <= radius
    return mask

#calculate flux threshold function parameters for different radiuses for each integrated image
def threshold_function_params_integrated_images(integrated_image_cube):
    integrated_std_function_params_array = []
    integrated_median_array = []
    for int_im in range(integrated_image_cube.shape[0]):
        print('calculating threshold: ',int_im+1,' out of ',integrated_image_cube.shape[0],end='\r')
        
        if background_noise_unifrom:    
            if int_im==0:
                integrated_image = integrated_image_cube[round(integrated_image_cube.shape[0]/2),:,:]
                popt_std, median = std_and_median_from_radius_function(integrated_image,cent_coord,image_radius)
            if int_im == integrated_image_cube.shape[0]-1:
                integrated_image = integrated_image_cube[int_im,:,:]
                popt_std, median= std_and_median_from_radius_function(integrated_image,cent_coord,image_radius)
           
            # calculate threshold function params
            integrated_median_array.append(median)
            integrated_std_function_params_array.append(popt_std)
         
        else:
            integrated_image = integrated_image_cube[int_im,:,:]         
            # calculate threshold function params
            popt_std, median= std_and_median_from_radius_function(integrated_image,cent_coord,image_radius)
            integrated_median_array.append(median)
            integrated_std_function_params_array.append(popt_std)
            
    return integrated_std_function_params_array, integrated_median_array


# integrate the cube slabs
def image_integrator(cube_image_data):

    if dithering: slab_length = int(round(int_image_length/2))
    else: slab_length = int_image_length
    #calculate how many integrated images
    cube_slab_number = np.floor_divide(cube_wavelength_range,slab_length)
    if np.mod(cube_wavelength_range,slab_length)!=0: cube_slab_number = cube_slab_number + 1

    integrated_image_cube = np.zeros((cube_slab_number,image_height,image_width))
    for int_im in range(cube_slab_number):
        print('integrating image: ',int_im+1,' out of ',cube_slab_number,end='\r')
 
        #get slab out of the cube
        if int_im == cube_slab_number-1: cube_slab = cube_image_data[ slab_length*int_im : wavelength_range, :, :] 
        else: cube_slab = cube_image_data[ slab_length*int_im : slab_length*(int_im+1)-1, :, :] 
      
        #create integrated image from cube slab
        integrated_image = np.nansum(np.array(cube_slab),0)
        integrated_image_cube[int_im,:,:] = integrated_image
    print('\n')
    if dithering:
        for int_im in range(cube_slab_number-1):
            print('integrating image: ',int_im+1,' out of ',cube_slab_number-1,end='\r')
            integrated_image_cube[int_im,:,:] = np.add(integrated_image_cube[int_im,:,:],integrated_image_cube[int_im+1,:,:])
        integrated_image_cube = integrated_image_cube[0:-1,:,:] # removes last item
        
    print('\n')
    print('integrated ',cube_slab_number,' images')
    return integrated_image_cube


def delete_proximate_sources_func(source_pixel_coords,source_channel_coords,min_pixel_dist,min_channel_dist):
    print('number of sources before deletion: ',len(source_pixel_coords))
    not_keep_coords = []
    for i, coord in enumerate(source_pixel_coords[:-1]):
        print('source: ',i+1,' out of ',len(source_pixel_coords),end='\r')
        
        x_dist = coord[0] - source_pixel_coords[i+1:,0]
        y_dist = coord[1] - source_pixel_coords[i+1:,1]
        channel_dist = np.abs(source_channel_coords[i] - source_channel_coords[i+1:])
        pixel_dist = np.sqrt(x_dist**2+y_dist**2)
        not_keep_coords.append(np.any((pixel_dist<min_pixel_dist) & (channel_dist< min_channel_dist)))
    keep_coords = np.logical_not(not_keep_coords)
    keep_coords = np.append(keep_coords,(True)) # keep the last element
    print('\n')
    print('number of sources after deletion: ',len(keep_coords[keep_coords==True]))
    return keep_coords


def sort_list(list_array,sorting_array):
    for array in range(len(list_array)):
        list_array[array]= (list_array[array])[np.argsort(sorting_array)]
    return list_array


def filter_list(list_array,keep_coords_array):
    for array in range(len(list_array)):
        list_array[array]= (list_array[array])[keep_coords_array]
    return list_array

# find and intially check the sources (>beamsize) in the integrated images
def search_integrated_images(integrated_image_cube, exclusion_zone_radius):
    cube_slab_number = integrated_image_cube.shape[0]
    
        
    for int_im in range(cube_slab_number):
        print('image: ',int_im+1,' out of ',cube_slab_number,end='\r')
        integrated_image = integrated_image_cube[int_im,:,:]
        int_image_wavelength = int_im*int_image_length
        if dithering: int_image_wavelength = int_im*int(round(int_image_length/2))
            
        #find the initial sources on the integrated image 
        popt_std = integrated_std_function_params_array[int_im]
        median = integrated_median_array[int_im]
        threshold_center = median + (exp_function(0.02,*popt_std))[0]*SNR_integrated_image_threshold
        tbl = find_peaks(integrated_image, threshold_center, box_size=exclusion_zone_radius )
        integrated_image_peaks_positions = np.transpose((tbl['x_peak'], tbl['y_peak']))
        max_source_value = tbl['peak_value']
    
        # check if the source max values are greater than threshold for the given radius      
        radiuses = np.sqrt( np.power(tbl['x_peak']-cent_coord[0],2) +  np.power(tbl['y_peak']-cent_coord[1],2))/image_radius
        threshold_for_each_source = median + exp_function(radiuses,*popt_std)*SNR_integrated_image_threshold
        integrated_image_peaks_positions = integrated_image_peaks_positions[max_source_value>=threshold_for_each_source]
        threshold_for_each_source_array = threshold_for_each_source[max_source_value>=threshold_for_each_source]     
        max_source_value = max_source_value[max_source_value>=threshold_for_each_source]
        
        # check if the found sources are larger than approximately the beamsize
        keep_source=[]
        initial_peaks_signal = np.array([])
        threshold_integrated_signal = np.array([])
        for source, source_coord in enumerate(integrated_image_peaks_positions):
            threshold = threshold_for_each_source_array[source]
            slit_beam = CircularAperture(source_coord, beam_radius[int_image_wavelength])
            mask= slit_beam.to_mask(method='center')
            slit_beam_image = mask.multiply(integrated_image) 
            beam_around_source = (np.array(slit_beam_image)).flatten()
            beam_around_source = beam_around_source[beam_around_source!=0]
            keep_source.append(len(beam_around_source[beam_around_source>0.5*threshold]) == len(beam_around_source)) 
            
            
        integrated_image_peaks_positions=integrated_image_peaks_positions[keep_source]
        max_source_value=max_source_value[keep_source]
        
      
        #save the coordinates and signal value for found sources
           
        if int_im == 0: 
            aperture_coordinates = integrated_image_peaks_positions
            max_source_value_array = max_source_value
            source_channel_coord_array = np.ones(len(integrated_image_peaks_positions))*(int_image_wavelength)
            
        else: 
            source_channel_coord_array = np.append(source_channel_coord_array,np.ones(len(integrated_image_peaks_positions))*int_image_wavelength)
            aperture_coordinates = np.append(aperture_coordinates, integrated_image_peaks_positions,axis=0)
            max_source_value_array = np.append(max_source_value_array,max_source_value)

    search_output = [aperture_coordinates, source_channel_coord_array]
    
    print('\n')
    print('total number of sources: ',len(aperture_coordinates))   
    return search_output

# check the spectrum of found systems (takes care of obvious continuum sources)
def check_spectrum(source_pixel_coords, source_channel_coords):
    print('starting spectral check')
    
    x0_array = np.zeros(len(source_pixel_coords))
    sigma_array = np.zeros(len(source_pixel_coords))
    A_array = np.zeros(len(source_pixel_coords))
    H_array = np.zeros(len(source_pixel_coords))
    rsqr_array = np.zeros(len(source_pixel_coords))
    SNR_spectrum_array = np.zeros(len(source_pixel_coords))
    spectrum_ok_array = np.zeros(len(source_pixel_coords))
    
    
    for source in range(len(source_pixel_coords)):
        print('source: ',source+1,' out of ',len(source_pixel_coords),end='\r')
        spectrum_ok=0
        
        # define wavelength ranges approximately around the found source
        wavelength_range = cube_image_data.shape[0]
        wavelength_range_array_flux = np.arange(source_channel_coords[source]-3*int_image_length,source_channel_coords[source]+4*int_image_length,1,dtype=int) 
        if source_channel_coords[source]-3*int_image_length <=0: wavelength_range_array_flux = np.arange(0,source_channel_coords[source]+4*int_image_length,1,dtype=int)
        if source_channel_coords[source]+4*int_image_length >= wavelength_range: wavelength_range_array_flux = np.arange(source_channel_coords[source]-3*int_image_length,wavelength_range,1,dtype=int)
        if source_channel_coords[source]+4*int_image_length >= wavelength_range and source_channel_coords[source]-3*int_image_length<=0 : wavelength_range_array_flux = np.arange(0,wavelength_range,1,dtype=int)
    
        # get the cube slab
        cube_slab_wavelength_range_flux = cube_image_data[wavelength_range_array_flux[0]:wavelength_range_array_flux[-1]+1][:][:]
         
        # get spectrum
        flux=np.zeros(len(wavelength_range_array_flux))
        for i, wavelength in enumerate(wavelength_range_array_flux):     
            image_slice = cube_slab_wavelength_range_flux[i][:][:] 
            slit = RectangularAperture([int((source_pixel_coords[source])[0]), int((source_pixel_coords[source])[1])], 3, 3)
            mask= slit.to_mask(method='center')
            slit_image = mask.multiply(image_slice) 
            flux[i] = np.nansum(slit_image)
                   
        # fit Gaussian to the spectral line
        xdata = wavelength_range_array_flux
        ydata = flux
        spectrum_fit,x0,sigma, A, H = fit_gauss(xdata,ydata)
        
        x0_array[source]=x0
        sigma_array[source]=sigma
        A_array[source]=A
        H_array[source]=H
        rsqr_array[source]=spectrum_fit
        SNR_spectrum_array[source]=(A/np.std(flux[round(x0+3*sigma):-1]))
        if np.isnan(SNR_spectrum_array[source]): SNR_spectrum_array[source]=(A/np.std(flux[0:round(x0-3*sigma)]))
        

        # verify the fit
        if spectrum_fit > rsqr_threshold: spectrum_ok = 1
    
        spectrum_ok_array[source] = spectrum_ok
    
    print(str(len(spectrum_ok_array[spectrum_ok_array==1])),' sources out of ',len(source_pixel_coords),'passed the spectral check')

    spectrum_ok_array = (spectrum_ok_array==1)
    spectrum_output_array = [rsqr_array, x0_array, sigma_array, A_array, H_array,SNR_spectrum_array]
    spectrum_output_array = filter_list(spectrum_output_array,spectrum_ok_array)
    
    return spectrum_ok_array, spectrum_output_array


def check_persistence(source_pixel_coords, source_channel_coords, source_frame_threshold,peak_zone_radius):
    print('starting persistence check')
    peak_zone_apertures = CircularAperture(source_pixel_coords, r=peak_zone_radius)
    
    persistence_ok_array = np.zeros(len(source_pixel_coords))
     
    for source in range(len(source_pixel_coords)):
        print('source: ',source+1,' out of ',len(source_pixel_coords),end='\r')
        signal_in_slice_position = np.array([])
            
        # define wavelength range approximately around the found source
        wavelength_range_array = np.arange(source_channel_coords[source]-1*int_image_length,source_channel_coords[source]+2*int_image_length,1,dtype=int) 
        if source_channel_coords[source]-1*int_image_length <=0: wavelength_range_array = np.arange(0,source_channel_coords[source]+2*int_image_length,1,dtype=int)
        if source_channel_coords[source]+2*int_image_length >= wavelength_range: wavelength_range_array = np.arange(source_channel_coords[source]-1*int_image_length,wavelength_range,1,dtype=int)
        if source_channel_coords[source]+2*int_image_length >= wavelength_range and source_channel_coords[source]-1*int_image_length<=0 : wavelength_range_array = np.arange(0,wavelength_range,1,dtype=int)
        cube_slab_wavelength_range = cube_image_data[wavelength_range_array[0]:wavelength_range_array[-1]+1][:][:]
       
        # find signal in peak zone image for each wavelength
        # creates signal_array (for exmaple [0 1 2 0 1 2 3 4 0 1 2 0 0 0 0 1 0]) marking at which wahelengths is the signal and how long it is persistent
        signal_length = 0
        signal_array = np.zeros(wavelength_range)
    
        for i, wavelength in enumerate(wavelength_range_array):
            # get threshold for the given radius and wavelength
            threshold = source_frame_threshold[source]
            # create peak zone image (circular image of r=peak_zone_radius centred on the target)
            image_slice = cube_slab_wavelength_range[i][:][:]  
            mask= peak_zone_apertures[source].to_mask(method='center')
            peak_zone_image = mask.multiply(image_slice, fill_value=np.nan)    
    
             # find peak, if no peak found above the threshold, error is returned
            try:          
                # find peak using np.max
                if np.nanmax(peak_zone_image)>= threshold: positions = np.unravel_index(np.nanargmax(peak_zone_image, axis=None), peak_zone_image.shape)
                else: np.make_error()
                positions = [[(positions[::-1])[0],(positions[::-1])[1]]]
               
                #check if the found source is at least beamsize, if not, error is returned
                slit_beam = CircularAperture(positions[0], beam_radius[wavelength])
                mask= slit_beam.to_mask(method='center')
                slit_beam_image = mask.multiply(peak_zone_image,fill_value=np.nan) 
                beam_around_source = (np.array(slit_beam_image)).flatten()
                beam_around_source = beam_around_source[np.invert(np.isnan(beam_around_source))]
                if len(beam_around_source[beam_around_source>0.5*threshold]) != len(beam_around_source): np.make_error()  
        
                # save the signal in the array
                signal_length = signal_length + 1
                signal_array[wavelength] = signal_length
        
                if i==0: signal_in_slice_position = posistions
                else: signal_in_slice_position = np.append(signal_in_slice_position,positions,axis=0)
                
            except:
                signal_array[wavelength] = 0
                signal_length = 0
                
                if i==0: signal_in_slice_position = [[0,0]]
                else: signal_in_slice_position = np.append(signal_in_slice_position,[[0,0]],axis=0)
    
        
        # verify if given signal is persistent and does not "jump around" too much        
        if np.max(signal_array)>signal_persistence_threshold:
            for i, wavelength in enumerate(wavelength_range_array):
                
                #passes only if wavelength is at the last element of a persistent signal
                if signal_array[wavelength]>2 and ( wavelength==len(signal_array)-1 or signal_array[wavelength+1]==0 ):
                    
                    signal_length = int(signal_array[wavelength])       
                    x_coords = signal_in_slice_position[i-signal_length+1:i+1][:,0]
                    y_coords = signal_in_slice_position[i-signal_length+1:i+1][:,1]
    
                    xdist = np.diff(x_coords)
                    ydist = np.diff(y_coords)
                    vector_length = np.sqrt(xdist**2+ydist**2)
                    ind = np.where(vector_length<max_signal_jump)
                    if len(np.diff(ind)[np.diff(ind)==1])>0: persistence_ok_array[source]=1

    print(str(len(persistence_ok_array[persistence_ok_array==1])),' sources out of ',len(source_pixel_coords),'passed the persistence check')
    persistence_ok_array = (persistence_ok_array==1)
    return persistence_ok_array
           

def calculate_spectral_SNR(source_pixel_coords, source_channel_coords):
    print('calculating spectral SNR')

    # trasnposed data cube
    transposed_cube_image_data = np.transpose(cube_image_data,(2,1,0))
    
    wavelength_range = cube_wavelength_range
    
    SNR_spectrum_array = np.zeros(len(source_pixel_coords)) 
    SNR_average_1_spectrum_array = np.zeros(len(source_pixel_coords))
    SNR_average_2_spectrum_array = np.zeros(len(source_pixel_coords))
    
    for source in range(len(source_pixel_coords)):
        print('source: ',source+1,' out of ',len(source_pixel_coords),end='\r')
        
        # define wavelength ranges approximately around the found source
        central_flux_left,central_flux_right = source_channel_coords[source]-1*int_image_length,source_channel_coords[source]+2*int_image_length
        if source_channel_coords[source]-1*int_image_length <=0: central_flux_left = 0
        if source_channel_coords[source]+2*int_image_length >= wavelength_range:  central_flux_right = wavelength_range

        outer_1_flux_left,outer_1_flux_right = source_channel_coords[source]-6*int_image_length,source_channel_coords[source]+7*int_image_length
        if source_channel_coords[source]-6*int_image_length <=0: outer_1_flux_left = 0
        if source_channel_coords[source]+7*int_image_length >= wavelength_range:  outer_1_flux_right = wavelength_range

        outer_2_flux_left,outer_2_flux_right = source_channel_coords[source]-10*int_image_length,source_channel_coords[source]+10*int_image_length
        if source_channel_coords[source]-10*int_image_length <=0: outer_2_flux_left = 0
        if source_channel_coords[source]+10*int_image_length >= wavelength_range:  outer_2_flux_right = wavelength_range

        central_flux_left,central_flux_right,outer_1_flux_left,outer_1_flux_right, outer_2_flux_left,outer_2_flux_right= int(central_flux_left),int(central_flux_right),int(outer_1_flux_left),int(outer_1_flux_right),int(outer_2_flux_left),int(outer_2_flux_right)
        
        # get spectrum
        x=round((source_pixel_coords[source])[0])
        y=round((source_pixel_coords[source])[1])
        flux_full = transposed_cube_image_data[x][y] 
        
        # central flux
        flux_central = flux_full[central_flux_left:central_flux_right]
        flux_central = flux_central[~np.isnan(flux_central)]
        
        # outer flux 1
        flux_outer_1 = np.append(flux_full[outer_1_flux_left:central_flux_left],flux_full[central_flux_right:outer_1_flux_right])
        flux_outer_1=flux_outer_1[~np.isnan(flux_outer_1)]
        
        # outer flux 2
        flux_outer_2 = np.append(flux_full[outer_2_flux_left:central_flux_left],flux_full[central_flux_right:outer_2_flux_right])
        flux_outer_2=flux_outer_2[~np.isnan(flux_outer_2)]
        
        # calculate spectral SNR
        flux_central=np.sort(flux_central)
        max1,max2,max3,max4,max5= flux_central[-1],flux_central[-2],flux_central[-3],flux_central[-4],flux_central[-5]
                    
        flux_outer_1=sigma_clip(flux_outer_1, sigma=3,  maxiters=5,masked=False)
        median = np.median(flux_outer_1)
        std = np.std(flux_outer_1)
        SNR1 = (max1-median)/std 
        SNR2 = (max2-median)/std 
        SNR3 = (max3-median)/std 
        SNR_average_1_spectrum_array[source] = (SNR1+SNR2+SNR3)/3
   
        flux_outer_2=sigma_clip(flux_outer_2, sigma=3,  maxiters=5,masked=False)
        median_2 = np.median(flux_outer_2)
        std_2 = np.std(flux_outer_2)
        SNR1 = (max1-median_2)/std_2 
        SNR2 = (max2-median_2)/std_2 
        SNR3 = (max3-median_2)/std_2 
        SNR_average_2_spectrum_array[source] = (SNR1+SNR2+SNR3)/3

        SNR_spectrum_array[source] = SNR1
    transposed_cube_image_data = None
    return SNR_spectrum_array,SNR_average_1_spectrum_array,SNR_average_2_spectrum_array


def associate_sources(source_pixel_coords, source_channel_coords, source_signal_strength, min_channel_dist, min_pixel_dist):
    associating_array = np.arange(0,len(source_pixel_coords))
    print('associating sources')
    print('number of sources before association: ',len(source_pixel_coords))
    source_pixel_coords = source_pixel_coords[np.argsort(source_channel_coords)]
    associating_array = associating_array[np.argsort(source_channel_coords)]
    source_signal_strength = source_signal_strength[np.argsort(source_channel_coords)]
    source_channel_coords = np.sort(source_channel_coords)
    
    # associate the sources
    source_as_array = np.zeros(len(source_pixel_coords))
    
    for i, coord in enumerate(source_pixel_coords):
        x_dist = coord[0] - source_pixel_coords[:,0]
        y_dist = coord[1] - source_pixel_coords[:,1]
        distances_angular = np.sqrt(x_dist**2+y_dist**2)
        distances_channel = np.abs(source_channel_coords[i] - source_channel_coords)
    
        if source_as_array[i]==0:
            source_as_array[(distances_angular < min_pixel_dist) & (distances_channel < min_channel_dist)] = i+1
            
        if source_as_array[i]!=0:
            source_as_array[(distances_angular < min_pixel_dist) & (distances_channel < min_channel_dist)] = source_as_array[i]
    
    # merge the associated sources

    associating_array = associating_array[np.argsort(source_signal_strength)]
    source_as_array =source_as_array[np.argsort(source_signal_strength)]

    strongest_in_system_id=[]
    for i,system in enumerate(np.unique(source_as_array)):
        strongest_in_system_id.append( ((np.arange(0,len(source_pixel_coords),1))[source_as_array==system])[-1] )
        
    associating_array = associating_array[strongest_in_system_id]
    print('number of sources after association: ',len(associating_array))
    return associating_array


def find_volumes(source_pixel_coords,source_channel,source_threshold):
    print('finding volumes of sources')
    xcent_array = np.zeros(len(source_pixel_coords))
    xmin_array = np.zeros(len(source_pixel_coords))
    xmax_array = np.zeros(len(source_pixel_coords))

    ycent_array = np.zeros(len(source_pixel_coords))
    ymin_array = np.zeros(len(source_pixel_coords))
    ymax_array = np.zeros(len(source_pixel_coords))

    zcent_array = np.zeros(len(source_pixel_coords))
    zmin_array = np.zeros(len(source_pixel_coords))
    zmax_array = np.zeros(len(source_pixel_coords))
    
    for source in range(len(source_pixel_coords)):
        print('source: ',source,end='\r')
        x_coord = int(round(source_pixel_coords[source][0]))
        y_coord = int(round(source_pixel_coords[source][1]))
        z_coord = int(round(source_channel[source]))
        if z_coord>cube_wavelength_range-1: z_coord=cube_wavelength_range-1
        threshold = source_threshold[source]
        volume_x_border_1 = 5
        volume_x_border_2 = 5
        
        volume_y_border_1 = 5
        volume_y_border_2 = 5
        
        volume_z_border_1 = 1
        volume_z_border_2 = 1
        
        fill_fraction = np.array([1,1,1,1,1,1])
        fill_fraction_threshold = 0.2
        loop=0
        while len(fill_fraction[(fill_fraction>fill_fraction_threshold)])!=0 and loop<200:
            loop+=1
            if z_coord-volume_z_border_1 <0: 
                fill_fraction[0]=0
                volume_z_border_1 = volume_z_border_1-1
            if z_coord+volume_z_border_2 > cube_wavelength_range-1: 
                fill_fraction[1]=0
                volume_z_border_2 = volume_z_border_2-1
            
                
            volume = cube_image_data[z_coord-volume_z_border_1:z_coord+volume_z_border_2,y_coord-volume_y_border_1:y_coord+volume_y_border_2,x_coord-volume_x_border_1:x_coord+volume_x_border_2]
            volume[np.isnan(volume)]=0
            fill_fraction = np.array([len((volume[0,:,:])[(volume[0,:,:]>threshold)])/len((volume[0,:,:]).flatten()),
                             len((volume[-1,:,:])[(volume[-1,:,:]>threshold)])/len((volume[-1,:,:]).flatten()),
                             len((volume[:,0,:])[(volume[:,0,:]>threshold)])/len((volume[:,0,:]).flatten()),
                             len((volume[:,-1,:])[(volume[:,-1,:]>threshold)])/len((volume[:,-1,:]).flatten()),
                             len((volume[:,:,0])[(volume[:,:,0]>threshold)])/len((volume[:,:,0]).flatten()),
                             len((volume[:,:,-1])[(volume[:,:,-1]>threshold)])/len((volume[:,:,-1]).flatten())])
            
            if fill_fraction[0] > fill_fraction_threshold: volume_z_border_1+=1
            if fill_fraction[1] > fill_fraction_threshold: volume_z_border_2+=1
            if fill_fraction[2] > fill_fraction_threshold: volume_y_border_1+=1
            if fill_fraction[3] > fill_fraction_threshold: volume_y_border_2+=1
            if fill_fraction[4] > fill_fraction_threshold: volume_x_border_1+=1
            if fill_fraction[5] > fill_fraction_threshold: volume_x_border_2+=1  
            if z_coord-volume_z_border_1 <0: 
                fill_fraction[0]=0
            if z_coord+volume_z_border_2 > cube_wavelength_range-1: 
                fill_fraction[1]=0

        xmin = x_coord-volume_x_border_1-10
        xmax = volume_x_border_2+x_coord+10
        xcent = int(round((xmax-xmin)/2)) +xmin

        ymin = y_coord-volume_y_border_1-10
        ymax = volume_y_border_2+y_coord+10
        ycent = int(round((ymax-ymin)/2))+ymin
        
        zmin = z_coord-volume_z_border_1-10
        zmax = volume_z_border_2+z_coord+10
        zcent = int(round((zmax-zmin)/2))+zmin

        xcent_array[source] = xcent
        xmin_array[source] = xmin
        xmax_array[source] = xmax
    
        ycent_array[source] = ycent
        ymin_array[source] = ymin
        ymax_array[source] = ymax
    
        zcent_array[source] = zcent
        zmin_array[source] = zmin
        zmax_array[source] = zmax
    find_volumes_output = [xcent_array,xmin_array,xmax_array,ycent_array,ymin_array,ymax_array,zcent_array,zmin_array,zmax_array]
    return find_volumes_output

def local_thresholds(source_coords,source_channel_coords):
    print('calculating local thresholds thresholds')
    SNR_array = np.zeros(len(source_coords))
    threshold_integrated_signal_array = np.zeros(len(source_coords))
    threshold_frame_signal_array = np.zeros(len(source_coords))
    max_source_value_array = np.zeros(len(source_coords))
    for source in range(len(source_coords)):
        print('source: ',source+1,' out of ',len(source_coords),end='\r')
        x_coord = source_coords[source][0]
        y_coord = source_coords[source][1]
        source_wavelength = source_channel_coords[source]
        if dithering: int_im = int(source_wavelength/(round(int_image_length/2)))
        else: int_im = int(source_wavelength/int_image_length)
    
        # integrated image threshold
        image_source = integrated_image_cube[int_im,y_coord-5:y_coord+5,x_coord-5:x_coord+5]
        image_source = image_source[~np.isnan(image_source)]
        max_source_value_array[source] = np.max(image_source)
        
        image = integrated_image_cube[int_im,y_coord-50:y_coord+50,x_coord-50:x_coord+50] 
        mask = np.full(image.shape,True)
        mask[40:60,40:60] = False
        image = image[mask]
        image = image[~np.isnan(image)]
        image=sigma_clip(image, sigma=3,  maxiters=5,masked=False)
        median = np.nanmedian(image)
        std = np.nanstd(image)
        threshold_integrated_signal_array[source] = median+SNR_integrated_image_threshold*std
        SNR_array[source] = (max_source_value_array[source] - median)/std
    
        # channel frame threshold
        frame_wavelength = int(source_wavelength)+5
        image_frame = cube_image_data[frame_wavelength,y_coord-50:y_coord+50,x_coord-50:x_coord+50]
        image_frame = image_frame[mask]
        image_frame = image_frame[~np.isnan(image_frame)]
        image_frame=sigma_clip(image_frame, sigma=3,  maxiters=5,masked=False)
        median = np.nanmedian(image_frame)
        std = np.nanstd(image_frame)
        threshold_frame_signal_array[source] = median+SNR_channel_frame_threshold*std
    keep_coords = (max_source_value_array>threshold_integrated_signal_array)
    return keep_coords,threshold_frame_signal_array,threshold_integrated_signal_array, SNR_array,max_source_value_array

In [6]:
# create folder for results
if not os.path.exists(path_to_results):
        os.makedirs(path_to_results)
    
#load in FITS file
cube_image_file = path_to_data + file
hdu_list = fits.open(cube_image_file)
hdu_list.info()
# cube_image_data = fits.getdata(cube_image_file , dtype='<f4')

# cube_image_data = hdu_list[0].data

cube_image_data = np.array((hdu_list[0].data)[cube_start:cube_end][:][:],dtype=np.float32)
print(cube_image_data.dtype)

# create WCS object for radio image
wcs_radio = WCS(naxis=2)
wcs_radio.wcs.crpix = [hdu_list[0].header['CRPIX1'], hdu_list[0].header['CRPIX2']]
wcs_radio.wcs.crval = [hdu_list[0].header['CRVAL1'], hdu_list[0].header['CRVAL2']]
wcs_radio.wcs.cunit = [hdu_list[0].header['CUNIT1'], hdu_list[0].header['CUNIT2']]
wcs_radio.wcs.ctype = [hdu_list[0].header['CTYPE1'], hdu_list[0].header['CTYPE2']]
wcs_radio.wcs.cdelt = [hdu_list[0].header['CDELT1'], hdu_list[0].header['CDELT2']]

wave0 = hdu_list[0].header['CRVAL3']
wavedelta = hdu_list[0].header['CDELT3']
channel0 = hdu_list[0].header['CRPIX3']

# image params
if cube_end == None: cube_end = hdu_list[0].header['NAXIS3']
cube_wavelength_range = cube_end - cube_start
wavelength_range = cube_end - cube_start
image_height = hdu_list[0].header['NAXIS1']
image_width  = hdu_list[0].header['NAXIS2']
cent_coord = (np.floor_divide(image_height,2),np.floor_divide(image_width,2))
image_radius = image_width/2

# beamsize table
beams_thruple = hdu_list[1].data
beams_thruple = beams_thruple[cube_start:cube_end]
beams_thruple = np.asarray(beams_thruple)
beam_diameter = np.zeros(len(beams_thruple))
for i in range(len(beams_thruple)):
    beam_diameter[i] = beams_thruple[i][0]
beam_diameter = np.round(beam_diameter/(3600*np.absolute(wcs_radio.wcs.cdelt[1])))
beam_diameter[np.mod(beam_diameter,2) == 0] = beam_diameter[np.mod(beam_diameter,2) == 0]-1
beam_radius = beam_diameter/2
median_beam_radius = np.median(beam_radius)
print(median_beam_radius)
# beam_radius = np.ones(cube_image_data.shape[0])*2.5
# median_beam_radius = np.median(beam_radius)

# close FITS file
hdu_list.close()

smallest_zone_radius = median_beam_radius*min_dist



Filename: ../data/MIGHTEE-HI_DR1_COSMOS_L2_r0p0_clean_conv_3001-4055.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      47   (4600, 4600, 1055)   float32   
  1  BEAMS         1 BinTableHDU     30   1055R x 5C   [1E, 1E, 1E, 1J, 1J]   
float32
2.5


In [7]:
# integrate images
integrated_image_cube = image_integrator(cube_image_data)

integrating image:  20  out of  20

integrating image:  19  out of  19

integrated  20  images


In [8]:
# calculate threshold params for integrated image  
integrated_std_function_params_array, integrated_median_array = threshold_function_params_integrated_images(integrated_image_cube)

In [9]:
# find smallest sources 
print('finding all sources')
aperture_coordinates, source_channel_coord_array = search_integrated_images(integrated_image_cube, smallest_zone_radius)

finding all sources
image:  19  out of  19

total number of sources:  49464


In [10]:
# recheck thresholds
keep_coords,threshold_frame_signal_array,threshold_integrated_signal_array,SNR_array,max_source_value_array = local_thresholds(aperture_coordinates,source_channel_coord_array)
print('\n')
print('number of sources left: ',len(keep_coords[keep_coords==True]))
list_found_sources = [aperture_coordinates, source_channel_coord_array, SNR_array,threshold_frame_signal_array,threshold_integrated_signal_array,max_source_value_array]
list_found_sources = filter_list(list_found_sources,keep_coords)

calculating local thresholds thresholds
source:  49464  out of  49464

number of sources left:  34098


In [11]:
# delete the integrated images to release memory
del integrated_image_cube

In [12]:
# delete extremely proximate (in distance and channel) sources
# this is aimed to delete sources that are likely to be the same sources that extend to different cube slabs
list_found_sources = sort_list(list_found_sources, list_found_sources[2])
keep_coords = delete_proximate_sources_func(list_found_sources[0],list_found_sources[1],5,2*int_image_length+1)
list_found_sources = filter_list(list_found_sources,keep_coords)


number of sources before deletion:  34098
source:  34097  out of  34098

number of sources after deletion:  27889


In [13]:
# check spectral SNR
SNR_spectrum_array,SNR_average_1_spectrum_array,SNR_average_2_spectrum_array = calculate_spectral_SNR(list_found_sources[0], list_found_sources[1])
spectral_SNR_ok_array = ((SNR_average_1_spectrum_array > SNR_spectrum_threshold)|(SNR_average_2_spectrum_array > SNR_spectrum_threshold))
list_found_sources=[*list_found_sources,SNR_spectrum_array]
list_found_sources = filter_list(list_found_sources,spectral_SNR_ok_array)

print('\n')
print('number of sources left: ',len(list_found_sources[0]))

calculating spectral SNR
source:  27889  out of  27889

number of sources left:  9291


In [14]:
# check persistence
persistence_ok_array = check_persistence(list_found_sources[0], list_found_sources[1], list_found_sources[3],median_beam_radius*2)
list_found_sources = filter_list(list_found_sources,persistence_ok_array)

starting persistence check
184  sources out of  9291 passed the persistence check


In [15]:
# check spectrum
spectrum_ok_array, list_spectrum_found_sources = check_spectrum(list_found_sources[0], list_found_sources[1])
list_found_sources = filter_list(list_found_sources,spectrum_ok_array)

starting spectral check
171  sources out of  184 passed the spectral check


In [16]:
# find the volumes of sources and get a better estimate of source centers
find_volumes_output = find_volumes(list_found_sources[0],list_spectrum_found_sources[1],list_found_sources[3])

# delete duplicates of volume centers
keep_coords = delete_proximate_sources_func(np.transpose((find_volumes_output[0],find_volumes_output[3])) ,find_volumes_output[6],10,5)

list_found_sources = filter_list(list_found_sources,keep_coords)
list_spectrum_found_sources = filter_list(list_spectrum_found_sources,keep_coords)
find_volumes_output = filter_list(find_volumes_output,keep_coords)
print('number of sources left: ',len(find_volumes_output[0]))

finding volumes of sources
number of sources before deletion:  171
source:  170  out of  171

number of sources after deletion:  171
number of sources left:  171


In [ ]:
# save results to a table
x_pix = find_volumes_output[0]
y_pix = find_volumes_output[3]
channel = find_volumes_output[6]+cube_start

xmin = find_volumes_output[1]
xmax = find_volumes_output[2]
ymin = find_volumes_output[4]
ymax = find_volumes_output[5]
zmin = find_volumes_output[7]+cube_start
zmax = find_volumes_output[8]+cube_start

SNR_array = (list_found_sources[2])
SNR_spectrum_array = (list_found_sources[6])
rsqr_array = (list_spectrum_found_sources[0])

x_pix_spec = np.transpose((list_found_sources[0]))[0]
y_pix_spec = np.transpose((list_found_sources[0]))[1]
x0_array= (list_spectrum_found_sources[1]) + cube_start
sigma_array = (list_spectrum_found_sources[2])
A_array = (list_spectrum_found_sources[3])
H_array = (list_spectrum_found_sources[4])


ra_deg, dec_deg = np.zeros(len(x_pix)),np.zeros(len(x_pix))
for i in range(len(x_pix)):
    ra_deg[i], dec_deg[i] = sky_coord_in_deg_from_pix(x_pix[i],y_pix[i],wcs_radio)
frequency = (channel_to_frequency(channel,wave0,wavedelta,channel0))

script_output = [ra_deg,dec_deg,frequency,x_pix,y_pix,channel,xmin,xmax,ymin,ymax,zmin,zmax,rsqr_array,SNR_array,SNR_spectrum_array,x_pix_spec,y_pix_spec,x0_array,sigma_array,A_array,H_array]
data = {'RA_deg': ra_deg,'DEC_deg': dec_deg,'frec [Hz]':frequency,'xpix': x_pix,'ypix': y_pix, 'channel':channel,'xpix_min':xmin,'xpix_max':xmax,'ypix_min':ymin,'ypix_max':ymax,'channel_min':zmin,'channel_max':zmax,'rsqr': rsqr_array,'integrated_SNR':SNR_array,'spectral_SNR':SNR_spectrum_array,'xpix_spec':x_pix_spec,'ypix_spec':y_pix_spec,'x0_channel': x0_array,'sigma_channel':sigma_array,'A':A_array,'H':H_array}

df = pd.DataFrame(data=data)
df.to_csv(path_to_results+'LESHI_found_sources_table.csv', index=False)  

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(df.to_string(index=False))
end = time.time()
print('runtime: ',end-start)


In [ ]:
print(len(list_found_sources[0]))

In [ ]:
# plot the results
rsqr_source = list_spectrum_found_sources[0]
source_coordinates = list_found_sources[0]

fig = plt.figure()
plt.figure(figsize = (5,5))
plt.imshow(np.zeros((4600,4600)), cmap='viridis', origin='lower', interpolation='nearest')
cmap = 'coolwarm_r'
norm = Normalize()
colors = plt.get_cmap(cmap)(norm(np.log10(rsqr_source)))

for i in range(len(source_coordinates)):
    CircularAperture(source_coordinates[i], r=25).plot(color=colors[i], lw=1.5)

fig.savefig(path_to_results+'final_sources.png')

In [ ]:
# create CARTA regions
source_coordinates_file = open(path_to_results+"source_regions.txt", "w")
source_coordinates_file.write('# Region file format: DS9 CARTA 4.1.0 \n')
source_coordinates_file.write('global color=green dashlist=8 3 width=1 font="helvetica 10 normal roman" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1 \n')
source_coordinates_file.write('image \n')

cmap = 'Reds_r'
norm = Normalize()
colors = plt.get_cmap(cmap)(norm(rsqr_array))
for i in range(len(rsqr_array)):
    color = matplotlib.colors.rgb2hex(colors[i])
    # color = matplotlib.colors.rgb2hex(0.5)
    
    source_coordinates_file.write('circle('+str((find_volumes_output[0])[i])+','+ str((find_volumes_output[3])[i])+','+ str(30)+')'+'# color='+str(color)+' width=2'+'\n')
    
source_coordinates_file.close()